In [ ]:
!git clone https://github.com/varsharao06/polyp-detection-safety-first.git


In [ ]:
import os
import shutil
import cv2
from sklearn.model_selection import train_test_split

# 1. Define source and destination directories
source_img_dir = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Original"
source_mask_dir = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Ground Truth"
base_dir = "/kaggle/working/yolo_data"

# Create YOLO directory structure
for split in ['train', 'val']:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)

# 2. Gather image files & split
images = sorted([f for f in os.listdir(source_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def process_split(file_list, split_name):
    for img_name in file_list:
        # Copy image
        src_img = os.path.join(source_img_dir, img_name)
        dst_img = os.path.join(base_dir, 'images', split_name, img_name)
        shutil.copy(src_img, dst_img)
        
        # Process corresponding mask to create YOLO segmentation txt
        mask_name = img_name # assuming identical filenames
        mask_path = os.path.join(source_mask_dir, mask_name)
        txt_path = os.path.join(base_dir, 'labels', split_name, f"{os.path.splitext(img_name)[0]}.txt")
        
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                h, w = mask.shape
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                with open(txt_path, 'w') as f:
                    for cnt in contours:
                        if cv2.contourArea(cnt) < 10:  # Ignore tiny noise artifacts
                            continue
                        # Format as YOLO segmentation: class_id x1 y1 x2 y2 ...
                        line = [0]
                        for point in cnt:
                            px, py = point[0]
                            line.append(px / w)
                            line.append(py / h)
                        f.write(" ".join(map(str, line)) + "\n")

process_split(train_imgs, 'train')
process_split(val_imgs, 'val')
print("Dataset split and label conversion complete!")

In [ ]:
from pathlib import Path

def get_class_ratio(labels_dir):
    labels_path = Path(labels_dir)
    images_dir = labels_path.parent.parent / "images" / labels_path.name
    
    image_files = list(images_dir.glob("*.*"))
    total_images = len(image_files)
    positive_images = 0
    negative_images = 0
    
    for img_file in image_files:
        label_file = labels_path / f"{img_file.stem}.txt"
        if label_file.exists() and label_file.stat().st_size > 0:
            positive_images += 1
        else:
            negative_images += 1
            
    print(f"Directory: {labels_path.name}")
    print(f"Total Images: {total_images}")
    print(f"Positive Images (with polyps): {positive_images}")
    print(f"Negative Images (empty): {negative_images}")
    if total_images > 0:
        print(f"Positive Ratio: {(positive_images / total_images) * 100:.2f}%\n")

get_class_ratio("/kaggle/working/yolo_data/labels/train")
get_class_ratio("/kaggle/working/yolo_data/labels/val")

In [ ]:
pip install ultralytics

In [ ]:
import os

# Check what folders exist inside your input dataset directory
dataset_path = "/kaggle/input/datasets/balraj98/cvcclinicdb" # or update if your dataset folder name differs
if os.path.exists(dataset_path):
    print("Contents of dataset directory:", os.listdir(dataset_path))
else:
    # Try listing the parent directory to find the exact folder name
    parent_dir = "/kaggle/input"
    print("Available inputs:", os.listdir(parent_dir))

In [ ]:
import os

# Walk through the dataset directory to find where the image files are located
dataset_root = "/kaggle/input/datasets/balraj98/cvcclinicdb"
for root, dirs, files in os.walk(dataset_root):
    img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if img_files:
        print(f"Found {len(img_files)} images in: {root}")

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. Point to the correct nested 'Original' image directory
source_img_dir = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Original"
base_dir = "/kaggle/working/yolo_data"

# Create YOLO directory structure
for split in ['train', 'val']:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)

# 2. Gather image files
images = sorted([f for f in os.listdir(source_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

# Split into train (80%) and val (20%)
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

# Function to copy images
def populate_split(file_list, split_name):
    for img_name in file_list:
        src_path = os.path.join(source_img_dir, img_name)
        dst_path = os.path.join(base_dir, 'images', split_name, img_name)
        shutil.copy(src_path, dst_path)

populate_split(train_imgs, 'train')
populate_split(val_imgs, 'val')

print(f"Successfully organized {len(train_imgs)} train and {len(val_imgs)} validation images!")

In [ ]:
import os
import shutil
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Define source and destination directories
source_img_dir = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Original"
source_mask_dir = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Ground Truth"
base_dir = "/kaggle/working/yolo_data"

# Create YOLO directory structure
for split in ['train', 'val']:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)

# 2. Gather image files & split
images = sorted([f for f in os.listdir(source_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def convert_mask_to_yolo(mask_path, txt_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return
    h, w = mask.shape
    
    # Find contours of the polyp mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    with open(txt_path, 'w') as f:
        for cnt in contours:
            if cv2.contourArea(cnt) < 10:  # Ignore tiny noise artifacts
                continue
            # Normalize polygon coordinates between 0 and 1
            segmentation = []
            for point in cnt:
                px, py = point[0]
                segmentation.append(px / w)
                segmentation.append(py / h)
            
            if len(segmentation) >= 6:  # Need at least 3 points for a polygon
                # Class 0 for 'polyp' followed by normalized polygon coordinates
                contour_str = " ".join([f"{coord:.6f}" for coord in segmentation])
                f.write(f"0 {contour_str}\n")

def populate_split(file_list, split_name):
    for img_name in file_list:
        # Copy Image
        src_img = os.path.join(source_img_dir, img_name)
        dst_img = os.path.join(base_dir, 'images', split_name, img_name)
        shutil.copy(src_img, dst_img)
        
        # Convert and Save Mask as YOLO Polygon Text File
        base_name, _ = os.path.splitext(img_name)
        mask_name = base_name + '.png' # Adjust if extension differs
        src_mask = os.path.join(source_mask_dir, img_name)
        
        if os.path.exists(src_mask):
            dst_txt = os.path.join(base_dir, 'labels', split_name, base_name + '.txt')
            convert_mask_to_yolo(src_mask, dst_txt)

populate_split(train_imgs, 'train')
populate_split(val_imgs, 'val')

print("Successfully converted PNG masks to YOLO polygon text annotations!")

In [ ]:
import yaml

# Create a valid yaml configuration pointing to the newly structured paths
data_config = {
    'path': '/kaggle/working/yolo_data',
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'polyp'}
}

config_path = '/kaggle/working/cvc_yolo.yaml'
with open(config_path, 'w') as f:
    yaml.dump(data_config, f)

# Start YOLOv8 Segmentation Training
from ultralytics import YOLO
model = YOLO("yolov8m-seg.pt")
model.train(data=config_path, epochs=50, imgsz=640)

In [ ]:
from ultralytics import YOLO

# Load your newly trained best weights
model = YOLO("/kaggle/working/runs/segment/train-10/weights/best.pt")

# Run prediction on a validation or test image
results = model.predict(source="/kaggle/working/yolo_data/images/val", save=True, conf=0.5)
print("Inference completed! Results saved.")